In [1]:
import pandas as pd
import torch
import os
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

In [2]:
input_file = "final_combined_dataset.csv"
output_file = "abstention_deepseek_final.csv"
save_every = 50
batch_size = 2

In [3]:
model_name = "deepseek-ai/deepseek-llm-7b-chat"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_8bit=True
)

model.eval()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(102400, 4096)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear8bitLt(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear8bitLt(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNor

In [4]:
df = pd.read_csv(input_file)

if os.path.exists(output_file):
    df_existing = pd.read_csv(output_file)
    start_idx = len(df_existing)
    print(f" Resuming from row {start_idx}")
else:
    start_idx = 0

In [5]:
def build_prompt(row):
    return f"""
You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT, DO NOT DEVIATE):
<answer>a/b/c/d/idk</answer>
<reasoning>Give 2-3 line explanation</reasoning>

Only output these two tags. No extra text.
"""

In [6]:
def parse_output(output):
    output_lower = output.lower()

    # IDK priority
    if "idk" in output_lower:
        final_answer = "idk"
    else:
        match = re.search(r"<answer>\s*(a|b|c|d)\s*</answer>", output_lower)
        final_answer = match.group(1) if match else "idk"

    reason_match = re.search(
        r"<reasoning>\s*(.*?)\s*</reasoning>",
        output,
        re.IGNORECASE | re.DOTALL
    )

    reasoning = reason_match.group(1).strip() if reason_match else ""

    if reasoning == "":
        final_answer = "idk"

    return final_answer, reasoning


In [ ]:
for i in tqdm(range(start_idx, len(df), batch_size), desc="Running Inference"):

    batch = df.iloc[i:i+batch_size].copy()
    prompts = [build_prompt(row) for _, row in batch.iterrows()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048   
    ).to(model.device)

    with torch.inference_mode():   
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,     
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None
        )

    input_len = inputs.input_ids.shape[1]
    generated_tokens = outputs[:, input_len:]

    decoded = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    pred_answers = []
    pred_reasoning = []

    for output in decoded:
        ans, reason = parse_output(output)
        pred_answers.append(ans)
        pred_reasoning.append(reason)

    batch["pred_final_answer"] = pred_answers
    batch["pred_reasoning"] = pred_reasoning

    if os.path.exists(output_file):
        batch.to_csv(output_file, mode="a", index=False, header=False)
    else:
        batch.to_csv(output_file, index=False)

    print(f"Saved batch {i} to {i + len(batch)}")

print("Done! Low-GPU pipeline completed.")

Running Inference:   0%|          | 0/1761 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
